# 01 — Exploratory Data Analysis

This notebook explores the Diabetic Retinopathy dataset:
- Class distribution analysis
- Patient-level statistics
- Sample image visualisation per class
- Image quality checks

In [ ]:
import importlib
import os
import sys

IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/diabetic_retinopathy"
    %pip install -q timm albumentations kaggle
else:
    PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.config
importlib.reload(src.config)
from src import config
from src.data_manager import ensure_dataset

ensure_dataset()
config.ensure_dirs()

print(f"Images dir: {config.EYEPACS_IMAGES_DIR} (exists: {os.path.isdir(config.EYEPACS_IMAGES_DIR)})")
print(f"Outputs dir: {config.OUTPUTS_DIR}")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import config
from src.utils import seed_everything

seed_everything()
sns.set_theme(style='whitegrid')
print(f'Labels CSV : {config.EYEPACS_LABELS_CSV}')
print(f'Images dir : {config.EYEPACS_IMAGES_DIR}')

## 1. Load and Inspect Labels

In [ ]:
df = pd.read_csv(config.EYEPACS_LABELS_CSV)
cols_to_drop = [c for c in df.columns if c.startswith('Unnamed')]
df = df.drop(columns=cols_to_drop, errors='ignore')

print(f'Total images : {len(df)}')
print(f'Columns      : {list(df.columns)}')
print(f'Missing vals : {df.isnull().sum().sum()}')
df.head(10)

In [ ]:
CLASS_CODE = dict(enumerate(config.CLASS_NAMES))
df['label_name'] = df['level'].map(CLASS_CODE)
df.describe()

## 2. Class Distribution

In [ ]:
CLASS_CODE = dict(enumerate(config.CLASS_NAMES))
class_counts = df['level'].value_counts().sort_index()
class_pct = (class_counts / len(df) * 100).round(1)

dist_df = pd.DataFrame({
    'Class': [CLASS_CODE[i] for i in class_counts.index],
    'Count': class_counts.values,
    'Percentage': class_pct.values,
})
print(dist_df.to_string(index=False))
print(f'\nImbalance ratio (max/min): {class_counts.max() / class_counts.min():.1f}x')

In [ ]:
CLASS_CODE = dict(enumerate(config.CLASS_NAMES))
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
colors = sns.color_palette('viridis', config.NUM_CLASSES)
bars = axes[0].bar(
    [CLASS_CODE[i] for i in range(config.NUM_CLASSES)],
    [class_counts.get(i, 0) for i in range(config.NUM_CLASSES)],
    color=colors, edgecolor='black', linewidth=0.5,
)
for bar, count in zip(bars, [class_counts.get(i, 0) for i in range(config.NUM_CLASSES)], strict=False):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{count:,}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Class Distribution', fontsize=14)
axes[0].set_xlabel('DR Grade')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(
    [class_counts.get(i, 0) for i in range(config.NUM_CLASSES)],
    labels=[CLASS_CODE[i] for i in range(config.NUM_CLASSES)],
    autopct='%1.1f%%', startangle=90, colors=colors,
    textprops={'fontsize': 10},
)
axes[1].set_title('Class Proportions', fontsize=14)

plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Patient-Level Analysis

In [ ]:
df['patient_id'] = df['image'].str.extract(r'(\d+)_')[0].astype(int)
df['eye'] = df['image'].str.extract(r'\d+_(\w+)')[0]

n_patients = df['patient_id'].nunique()
n_images = len(df)
images_per_patient = df.groupby('patient_id').size()

print(f'Unique patients       : {n_patients:,}')
print(f'Total images          : {n_images:,}')
print(f'Avg images per patient: {images_per_patient.mean():.2f}')
print('Eye distribution      :')
print(df['eye'].value_counts())

In [ ]:
# Check for patients where left and right eyes have different grades
patient_grades = df.groupby('patient_id')['level'].agg(['min', 'max'])
discordant = patient_grades[patient_grades['min'] != patient_grades['max']]
print(f'Patients with different grades for left vs right eye: {len(discordant)} / {n_patients}')
print(f'This is {100 * len(discordant) / n_patients:.1f}% of patients')

## 4. Sample Images per Class

In [ ]:
CLASS_CODE = dict(enumerate(config.CLASS_NAMES))
from src import config


def find_image(images_dir, image_id):
    """Try common extensions and return the loaded RGB image, or None."""
    for ext in ('.jpeg', '.jpg', '.png', '.JPEG', '.JPG'):
        p = os.path.join(images_dir, image_id + ext)
        if os.path.isfile(p):
            img = cv2.imread(p)
            if img is not None:
                return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return None

# Reload config to pick up nested-folder detection
importlib.reload(src.config)

# Diagnostic
print(f'Images dir: {config.EYEPACS_IMAGES_DIR}')
print(f'  exists: {os.path.isdir(config.EYEPACS_IMAGES_DIR)}')
if os.path.isdir(config.EYEPACS_IMAGES_DIR):
    all_files = os.listdir(config.EYEPACS_IMAGES_DIR)
    print(f'  total files: {len(all_files)}')
    print(f'  sample files: {all_files[:5]}')
else:
    print('  ERROR: directory not found!')

n_samples = 5
fig, axes = plt.subplots(config.NUM_CLASSES, n_samples, figsize=(3 * n_samples, 3 * config.NUM_CLASSES))
fig.suptitle('Sample Retinal Images per DR Grade', fontsize=18, y=1.02)

not_found = 0
for cls in range(config.NUM_CLASSES):
    cls_df = df[df['level'] == cls]
    sampled = cls_df.sample(n=min(n_samples, len(cls_df)), random_state=config.SEED)

    for j, (_, row) in enumerate(sampled.iterrows()):
        img = find_image(config.EYEPACS_IMAGES_DIR, row['image'])
        if img is not None:
            axes[cls, j].imshow(img)
        else:
            not_found += 1
        axes[cls, j].axis('off')
        if j == 0:
            axes[cls, j].set_ylabel(CLASS_CODE[cls], fontsize=12, rotation=0, labelpad=80)

if not_found > 0:
    print(f'WARNING: {not_found}/{n_samples * config.NUM_CLASSES} images not found!')

plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, 'sample_images_per_class.png'), dpi=300, bbox_inches='tight')
plt.show()

## 5. Image Quality Check

In [ ]:
# Sample a subset for quality analysis
sample_size = min(500, len(df))
quality_sample = df.sample(n=sample_size, random_state=config.SEED)

heights, widths, brightness_vals, contrast_vals = [], [], [], []
corrupt_count = 0

for _, row in quality_sample.iterrows():
    img = find_image(config.EYEPACS_IMAGES_DIR, row['image'])
    if img is None:
        corrupt_count += 1
        continue
    h, w = img.shape[:2]
    heights.append(h)
    widths.append(w)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    brightness_vals.append(gray.mean())
    contrast_vals.append(gray.std())

print(f'Sampled: {sample_size} | Corrupt/missing: {corrupt_count}')
if heights:
    print(f'Height  — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')
    print(f'Width   — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')
    print(f'Bright  — min: {min(brightness_vals):.1f}, max: {max(brightness_vals):.1f}, mean: {np.mean(brightness_vals):.1f}')
    print(f'Contrast— min: {min(contrast_vals):.1f}, max: {max(contrast_vals):.1f}, mean: {np.mean(contrast_vals):.1f}')
else:
    print('ERROR: No images loaded. Check config.EYEPACS_IMAGES_DIR path.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(brightness_vals, bins=40, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Brightness Distribution (sample)', fontsize=13)
axes[0].set_xlabel('Mean Pixel Intensity')
axes[0].set_ylabel('Count')

axes[1].hist(contrast_vals, bins=40, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('Contrast Distribution (sample)', fontsize=13)
axes[1].set_xlabel('Pixel Std Dev')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, 'image_quality.png'), dpi=150, bbox_inches='tight')
plt.show()

## APTOS 2019 Dataset (Stage 2)

The APTOS 2019 dataset is used for Stage 2 fine-tuning. It has ~3,600 images
with cleaner labels from trained clinicians. Let's compare it with EyePACS.


In [ ]:
# Load APTOS dataset
aptos_csv = os.path.join(config.APTOS_DATA_DIR, "train.csv")
if os.path.isfile(aptos_csv):
    aptos_df = pd.read_csv(aptos_csv)
    aptos_df = aptos_df.rename(columns={"id_code": "image", "diagnosis": "level"})
    print(f"APTOS images: {len(aptos_df)}")
    print(f"\nClass distribution:")
    print(aptos_df["level"].value_counts().sort_index())
    
    # Side-by-side comparison
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle("Class Distribution: EyePACS vs APTOS", fontsize=16, y=1.02)
    
    CLASS_CODE = dict(enumerate(config.CLASS_NAMES))
    for ax, (name, data) in zip(axes, [("EyePACS", df), ("APTOS", aptos_df)]):
        counts = data["level"].value_counts().sort_index()
        colors = sns.color_palette("viridis", config.NUM_CLASSES)
        bars = ax.bar(
            [CLASS_CODE[i] for i in range(config.NUM_CLASSES)],
            [counts.get(i, 0) for i in range(config.NUM_CLASSES)],
            color=colors, edgecolor="black", linewidth=0.5,
        )
        ax.set_title(f"{name} (n={len(data):,})", fontsize=13, fontweight="bold")
        ax.set_ylabel("Count")
        ax.tick_params(axis="x", rotation=15)
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width()/2, h, f"{int(h)}", 
                        ha="center", va="bottom", fontsize=8)
    
    plt.tight_layout()
    plt.savefig(os.path.join(config.FIGURES_DIR, "class_distribution_comparison.png"), dpi=200, bbox_inches="tight")
    plt.show()
    
    # Summary table
    summary = pd.DataFrame({
        "EyePACS": df["level"].value_counts().sort_index(),
        "APTOS": aptos_df["level"].value_counts().sort_index(),
    })
    summary.index = [CLASS_CODE[i] for i in summary.index]
    summary["EyePACS %"] = (summary["EyePACS"] / summary["EyePACS"].sum() * 100).round(1)
    summary["APTOS %"] = (summary["APTOS"] / summary["APTOS"].sum() * 100).round(1)
    print("\n" + summary.to_string())
else:
    print(f"APTOS dataset not found at {aptos_csv}. Skipping.")
    print("Download with: python train_pipeline.py (auto-downloads from Kaggle)")


## 6. Summary

Key findings:
- **Severe class imbalance** — Class 0 (No DR) dominates at ~73%, Severe and Proliferative are each ~2-5%
- Patient-level splitting is essential — images named `{patient_id}_{left|right}`
- A significant portion of patients have discordant grades between eyes
- Image quality varies — preprocessing (CLAHE) will help normalise